# Set up

## Import packages

In [ ]:
import pandas as pd
import numpy as np
import pyspark.sql.functions as sf

## Locate lakehouse
Using the `notebookutils` package we can locate the lake house in our current workspace, this allows for the path to dynamically change depending on the workspace it is on, this then allows the use of a CI/CD git pipeline where a development workspace associated with the `Staging` branch seamlessly integrates with the production workspace associated with the `main` branch.

In [ ]:
notebookutils.credentials.help()

In [ ]:
storage_workspace_id = "29cfe25d-afd5-473a-91d8-1a13b9efdddf"
lakehouse_artifact = notebookutils.lakehouse.get(name="DataLakehouse", workspaceId=storage_workspace_id)

lakehouse_id = lakehouse_artifact.id

## Load raw data

To load the data we use a f-string to format all of our dynamic IDs and our selected schema and table into a ABFS (Azure Blob File System) path. These are then read as delta tables and converted from spark to pandas dataframes.

In [ ]:
lakehouse_schema = "RawData"
path_to_schema = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}/"

# PIS data
raw_digital_df = spark.read.format("delta").load(path_to_schema + "Raw_DigitalCustomerData").toPandas()
raw_paper_df = spark.read.format("delta").load(path_to_schema + "Raw_PaperCustomerData").toPandas()

# DTU data
courses_df = spark.read.format("delta").load(path_to_schema + "Raw_CourseData").toPandas()
students_df = spark.read.format("delta").load(path_to_schema + "Raw_StudentData").toPandas()
student_admission_df = spark.read.format("delta").load(path_to_schema + "Raw_StudentAdmissionByGenderData").toPandas()
study_program_df = spark.read.format("delta").load(path_to_schema + "Raw_StudyProgramData").toPandas()

# Preprocess Data 

## Customer Data

### Prepare the paper data
Create new `TimeStamp` column that combines the `Date` and the `Slot1/Slot2` columns into a comprehensive datetime column, after which the aford mentioned columns are dropped.

In [ ]:
datetime_format = r"%d/%m/%Y %H:%M:%S"
date_format = r"%d/%m/%Y"
date_column = pd.to_datetime(raw_paper_df['Date'],          format=date_format).dt.date.astype(str)
time_column = pd.to_datetime(raw_paper_df['Slot1/Slot2'],   format=datetime_format).dt.time.astype(str)

raw_paper_df['TimeStamp'] = pd.to_datetime(date_column + ' ' + time_column)
raw_paper_df = raw_paper_df.drop(columns=['Date','Slot1/Slot2'])

The first few hundred entries of the paper customer data come as daily aggregates of the number of visitors, this are then selected and their `Number_visitor` column is used to repeat the entries as meany times as the value in the column states, yielding an expanded version of this initial rudimentary data where each visitor has an entry, this allows it to be combined with all subsequent data as all the processing and visualization logic works base on the premise that there are individual entries for each customer.

In [ ]:
multi_entry_df = raw_paper_df[raw_paper_df["Rating"].notna() & raw_paper_df["Number_visitor"].notna()]
expanded_multi_entry_df = multi_entry_df.loc[multi_entry_df.index.repeat(multi_entry_df['Number_visitor'])].drop(columns='Number_visitor').reset_index(drop=True)

Lastly, the single entry and the expanded multi entry dataframes are concatenated into one.

In [ ]:
single_entry_df = raw_paper_df[raw_paper_df["Rating"].notna() & raw_paper_df["Number_visitor"].isna()].drop(columns='Number_visitor')
combined_paper_df = pd.concat([expanded_multi_entry_df, single_entry_df], ignore_index=True)

### Column naming cohesion
Rename columns in the raw digital and combined paper dataframes to non space separated, non capitalized variants for later concatenation

In [ ]:
raw_digital_df = raw_digital_df.rename(
    columns={
        "Completion time"                   : "time_stamp",
        "Enter you student number"          : "s_number",
        "Did you get the help you needed?"  : "rating",
        "Course Number"                     : "course_number",
        "Location"                          : "location",
        "Workshop"                          : "workshop_attendance"
        })

combined_paper_df = combined_paper_df.rename(
    columns={
        "Study number"                      : "s_number",
        "Location"                          : "location",
        "Rating"                            : "rating",
        "TimeStamp"                         : "time_stamp",
        })

### Aggregate paper and digital data

Finally, the 2 data sets are concatenated creating and aggregated customer dataframe. Its blank spaces are then filled with numpy `NaN` except for the `workspace_attendance` column which is filled with `False` for all entries missing a value. Lastly, appropriate data types are set on each column.

In [ ]:
# Concatenate data frames
aggregated_data_df = pd.concat([combined_paper_df,raw_digital_df], ignore_index=True)

#Fill workshop column NaNs with False
aggregated_data_df['workshop_attendance'] = aggregated_data_df['workshop_attendance'].fillna(value=False)

#Fill blank spaces with `np.NaN`
aggregated_data_df = aggregated_data_df.replace(r'^\s*$', np.nan, regex=True).fillna("N/A")

# Final type setting for column
aggregated_data_df['time_stamp'] = pd.to_datetime(aggregated_data_df['time_stamp'], format=datetime_format)
aggregated_data_df = aggregated_data_df.astype({'s_number': str, 'location': str, 'rating': int, 'course_number': str, 'workshop_attendance': bool})

## Clean and rename DTU data

### Course DB

Initially, the raw courses DB uses very non-descriptive column names, during this step they are renames to `course_number` and `course_name`.

In [ ]:
courses_df = courses_df.rename(
    columns={
        "Name"          :   "course_number",
        "Value.name"    :   "course_name",
    })

### Students DB

Similarly the student DB initially comes with columns which are either over complicated, unnecessary or in danish. Therefore, for simplicity they are renamed and reduced to `gender`, `name`, `s_number`, `direction_EN`, `direction_DK`, `education_EN` and `education_DK`.

In [ ]:
# Rename columns
students_df = students_df.rename(
    columns={
        "KoenFraCpr"        :   "gender",
        "displayName"       :   "name",
        "stads_studentcode" :   "s_number",
        "retning_uk"        :   "direction_EN",
        "uddannelse_uk"     :   "education_EN",
        "retning_dk"        :   "direction_DK",
        "uddannelse_dk"     :   "education_DK",
        })

students_df = students_df.drop(columns=['study_frame', 'study_frame2'])

Furthermore, the `gender` and `education_EN` columns are remapped to use more significant and simpler parameters. Aswell as filling any missing values with `Other`.

In [ ]:
gender_map = {'M': 'Male', 'K': 'Female'}
students_df['gender'] = students_df['gender'].map(gender_map).fillna('Other')

edu_map = {
    'Bachelor in Engineering': 'Diplom / BEng',
    'Master of Science in Engineering': 'MSc in Engineering',
    'BSc in Engineering': 'BSc in Engineering'
}
students_df['education_EN'] = students_df['education_EN'].map(edu_map).fillna('Other')

### Study Programs DB

Firstly, the `Education` and `Direction` column values are mapped to values that match the student database. This allows for later crossreferencing of the data.

In [ ]:
study_program_df['Education'] = (study_program_df['Education']
    .str.replace('Civilbachelor', 'BSc in Engineering', regex=False)
    .str.replace('Diplomingeniør', 'Diplom / BEng', regex=False)
)

direction_map = {
    "Computer Engineering": "Computer eng.",
    "Kemi og Teknologi": "Kemi og tek.",
    "Arktisk Byggeri og Infrastruktur": "Arktisk tek.",
    "Byggeri og Infrastruktur": "Byggeri og infrast.",
    "Elektrisk Energiteknologi": "Elektrisk energitek.",
    "Elektroteknologi": "Elektrotek.",
    "Fødevaresikkerhed og -kvalitet": "Fødevareanaly.",
    "Global Business og Teknologi": "Global Biz og tek.",
    "Kemi- og Bioteknik": "Kemi/biotek.",
    "Kemiteknik og International Business": "Kemitek./int. business",
    "Maritim Teknik": "Maritim tek.",
    "Mobilitet, Transport og Logistik": "Mobilitet/transport",
    "Softwareteknologi": "Softwaretek.",
    "Sundhedsteknologi": "Sundhedstek.",
    "Anvendt Matematik": "Matematik og tek",
    "Byggeteknologi": "Byggetek.",
    "Cyberteknologi": "Cybertek.",
    "Design og Innovation": "Design og inn.",
    "Fysik og Ingeniørvidenskab": "Fysik og ing.videnskab",
    "Geofysik og Rumteknologi": "Geofysik og rumtek.",
    "Kvan. Biologi og Sygdomsmodel. (2020 og før)": "Kvantitativ biologi",
    "Life Science og Teknologi": "Life Science",
    "Medicin og Teknologi": "Medicin og tek",
    "Miljøteknologi": "Miljøtek.",
    "Autonome Systemer": "Autonome sys.",
    "Bioinformatik": "Bioinfo. og systembio.",
    "Bæredygtig Energi": "Bæredygtig energi",
    "Bæredygtige Energisystemer": "Bæredygtig energisys.",
    "Bæredygtige Energiteknologier": "Bæredygtige energitek.",
    "Bæredygtigt Fiskeri og Akvakultur": "Bæredygtigt fisk. og akva.",
    "Biomaterialeteknologier til Medicin": "Biomaterialeteknologier til med.",
    "Farmateknologi": "Farmatek.",
    "Fysik og Teknologi": "Fysik og tek.",
    "Fødevareteknologi": "Fødevaretek.",
    "Industriel Økonomi og Teknologiledelse": "Industriel Øko. og Tekn. ledelse",
    "Informationsteknologi": "Informationstek.",
    "Kemisk og Biokemisk Teknologi": "Kemisk og Biokemisk Tek.",
    "Kommunikationsteknologier og Systemdesign": "Kom.tek. og system",
    "Konstruktion og Mekanik": "Konstruktion og mek.",
    "Kvantitativ Biologi og Sygdomsmodellering": "Kvant. Biologi og Sygdomsmod.",
    "Lyd og Akustisk Teknologi": "Lyd og Akustisk Tek.",
    "Lys og Teknologi": "Lys og tek.",
    "Matematisk Modellering og Computing": "Mat. model. og computing",
    "Materiale- og Procesteknologi": "Materiale og procestek.",
    "Medicin og tek": "Medicin og tek.",
    "Menneskeorienteret Kunstig Intelligens": "Menneskeorienteret AI",
    "Teknologisk Entreprenørskab": "Tek. entreprenørskab",
}

study_program_df['Direction'] = study_program_df['Direction'].replace(direction_map)

Lastly, the year columns of the number of students enrolled are filled with 0's on all the blanks, `-` or `*`.

In [ ]:
year_columns = study_program_df.columns.drop(['Education', 'Direction'])

study_program_df[year_columns] = study_program_df[year_columns].replace('-', '0', regex=False).replace('*', '0', regex=False)

study_program_df[year_columns] = study_program_df[year_columns].apply(pd.to_numeric, errors='raise')

# Save data to Lakehouse

In [ ]:
# Set up lakehouse destination parameters. Create schema if missing
lakehouse_schema = 'PreProcessedData'
path_to_schema = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}"

if notebookutils.fs.exists(path_to_schema):
    notebookutils.fs.mkdirs(path_to_schema)

In [ ]:
# Conversion to spark dataframe and preparation for saving.
aggregated_data = spark.createDataFrame(aggregated_data_df)
student_data = spark.createDataFrame(students_df)
course_data = spark.createDataFrame(courses_df)
study_program_data = spark.createDataFrame(study_program_df)

#### Aggregated customer data

In [ ]:
table_name = 'Aggregated_CustomerData'
abfss_path = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}/{table_name}"

aggregated_data.write.mode("overwrite").option("mergeSchema", "true").format("delta").save(abfss_path)

#### Course data

In [ ]:
table_name = 'Preproccesed_CourseData'
abfss_path = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}/{table_name}"

course_data.write.mode("overwrite").option("mergeSchema", "true").format("delta").save(abfss_path)

#### Student data

In [ ]:
table_name = 'Preproccesed_StudentData'
abfss_path = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}/{table_name}"

student_data.write.mode("overwrite").option("mergeSchema", "true").format("delta").save(abfss_path)

#### Study program data

In [ ]:
table_name = 'Preproccesed_StudyProgramData'
abfss_path = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}/{table_name}"

study_program_data.write.mode("overwrite").option("mergeSchema", "true").format("delta").save(abfss_path)